<a href="https://colab.research.google.com/github/mf2056/F20AA/blob/main/DataAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install pandas nltk vaderSentiment scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 6.5 MB/s eta 0:00:00


In [2]:
from google.colab import files
uploaded = files.upload()

Saving youtube_amazon_employee.csv to youtube_amazon_employee.csv


In [66]:
import pandas as pd
df = pd.read_csv("youtube_amazon_employee.csv")
df.head()

,text,platform,source
0,Currently working at amazon. I work Front half...,youtube,Lyqo2uJvNO4
1,Going on 8.5 years,youtube,Lyqo2uJvNO4
2,Do they have a union? Jeeze…,youtube,Lyqo2uJvNO4
3,Workers having sex in the bathrooms at amazon.,youtube,Lyqo2uJvNO4
4,I work sort. The day I quit Im fucking the sor...,youtube,Lyqo2uJvNO4


In [67]:
#Basic Cleaning

comment_col = "text"
df = df.dropna(subset=[comment_col])  # Drop rows with missing comments
df[comment_col] = df[comment_col].astype(str).str.strip()

# Remove empty strings
df = df[df[comment_col] != ""]

# Remove duplicates
df = df.drop_duplicates(subset=[comment_col])

# Remove very short comments (less than 3 words)
df["comment_word_count"] = df["text"].astype(str).str.split().str.len()
df = df[df["comment_word_count"] >= 3]

df.shape

(2803, 4)

In [68]:
# Remove spam comments

spam_keywords = [
    "subscribe", "giveaway", "click", "telegram", "whatsapp",
    "contact me", "dm me", "crypto", "bitcoin", "forex", "trading",
    "please like", "anyone watching in", "bit.ly", "goo.gl",
    "link in bio", "visit my site"
]

def remove_spam(text):
    text_lower = text.lower()
    return not any(word in text_lower for word in spam_keywords)

df = df[df[comment_col].apply(remove_spam)]
df.shape

(2787, 4)

In [69]:
# Filtering relevant comments

workplace_keywords = [
    "employee", "worker", "staff", "associate", "warehouse",
    "leadership", "executive", "perks", "insurance",
    "fulfillment", "manager", "management", "boss",
    "hr", "supervisor", "shift", "overtime", "break", "pay",
    "salary", "wage", "benefits", "culture", "toxic", "pressure",
    "workload", "stress", "burnout", "union", "treatment",
    "working", "job", "career", "fired", "hired",
    "workplace", "company", "office", "corporate", "hiring",
    "recruitment", "interview", "promotion", "resignation",
    "quit", "quitting", "layoff",
]

def is_relevant(text):
    text_lower = text.lower()
    return any(word in text_lower for word in workplace_keywords)

df = df[df[comment_col].apply(is_relevant)]
df.shape

(1416, 4)

In [71]:
# Sentiment Analysis

from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()

def get_vader_scores(text):
    return analyzer.polarity_scores(text)

df["vader_scores"] = df[comment_col].apply(get_vader_scores)

df["compound"] = df["vader_scores"].apply(lambda x: x["compound"])
df["pos_score"] = df["vader_scores"].apply(lambda x: x["pos"])
df["neu_score"] = df["vader_scores"].apply(lambda x: x["neu"])
df["neg_score"] = df["vader_scores"].apply(lambda x: x["neg"])

df.head()


,text,platform,source,comment_word_count,vader_scores,compound,pos_score,neu_score,neg_score
0,Currently working at amazon. I work Front half...,youtube,Lyqo2uJvNO4,593,"{'neg': 0.057, 'neu': 0.798, 'pos': 0.145, 'co...",0.9959,0.145,0.798,0.057
2,Do they have a union? Jeeze…,youtube,Lyqo2uJvNO4,6,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000
3,Workers having sex in the bathrooms at amazon.,youtube,Lyqo2uJvNO4,8,"{'neg': 0.0, 'neu': 0.805, 'pos': 0.195, 'comp...",0.1779,0.195,0.805,0.000
4,I work sort. The day I quit Im fucking the sor...,youtube,Lyqo2uJvNO4,13,"{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound...",0.0000,0.000,1.000,0.000
5,From my experience with talking to many associ...,youtube,Lyqo2uJvNO4,74,"{'neg': 0.108, 'neu': 0.818, 'pos': 0.073, 'co...",-0.6002,0.073,0.818,0.108


In [75]:
# Convert Score to Label

def label_sentiment(compound):
    if compound >= 0.02:
        return "positive"
    elif compound <= -0.02:
        return "negative"
    else:
        return "neutral"

df["sentiment_label"] = df["compound"].apply(label_sentiment)

df["sentiment_label"].value_counts()

# See only the first 10 comments with VADER scores and labels
df[["text", "sentiment_label"]].head(20)

,text,sentiment_label
0,Currently working at amazon. I work Front half...,positive
2,Do they have a union? Jeeze…,neutral
3,Workers having sex in the bathrooms at amazon.,positive
4,I work sort. The day I quit Im fucking the sor...,neutral
5,From my experience with talking to many associ...,negative
6,I experienced a similar experience working at ...,negative
7,I worked there for five years bust my ass they...,negative
8,Video title should be “Grown men guides to be ...,negative
9,I remember when I quit. Very clearly. I had as...,negative
11,"Actually, if you use the bathroom on yourself ...",neutral


In [31]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment_label"]
)

print("Train size:", train_df.shape)
print("Test size:", test_df.shape)

train_df["sentiment_label"].value_counts(normalize=True)

Train size: (1124, 9)
Test size: (282, 9)


,proportion
sentiment_label,
positive,0.525801
negative,0.338078
neutral,0.136121
